# 🧬 Notebook 0 (Optional): Build a Dataset From Any ChIP-seq Peaks File + Reference Genome

**How to read this notebook:**
### How to read this notebook

| Marker | Meaning |
|---|---|
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it. |
| 👀 **READ** | Important code. Read the comments and follow the main idea. |
| 🧠 **BUILD IT** | A core concept turned into code. Read this one closely. |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves. |
| 🔲 **YOUR TURN** | A line is left blank on purpose. Write it, then run the check cell below it. |
| ✅ **CHECKPOINT** | Stop and answer before moving on. |

Every notebook in this bootcamp uses these same six markers.

**Why this notebook exists:** Notebooks 1-3 use a fixed dataset at
`~/ctcf_k562_example/`. This notebook shows how that dataset gets BUILT from
two raw inputs — a peaks file and a reference genome — so you can swap in a
different transcription factor, cell type, or species.

**Roadmap**
1. What peaks files and reference genomes actually contain
2. Environment setup
3. 🎯 Configure your dataset (the one section you'll usually edit)
4. Load the reference genome
5. Load and clean the peaks file
6. Extract positive (binding) sequences — **includes an important data-quality fix**
7. Generate background (negative) sequences
8. Write out seqs.txt / labels.txt + a config file
9. Run a full dataset sanity-check suite
10. Point Notebooks 1-3 at your new dataset

## 🧭 Python survival guide — read this once, then come back when needed

You do **not** need to memorize Python syntax. When you see unfamiliar code, first identify the job it is doing.

| Python word | Plain-English meaning | Tiny example |
|---|---|---|
| **variable** | A name that stores a value | `k = 6` |
| **function** | A reusable mini-program that performs one job | `gc_content(sequence)` |
| **argument** | A value you give to a function | `gc_content("ACGT")` |
| **return** | The value a function gives back | `return gc_fraction` |
| **list** | An ordered collection | `["A", "C", "G", "T"]` |
| **dictionary (`dict`)** | Named values stored as key → value pairs | `{"A": 1, "C": 2}` |
| **DataFrame** | A pandas table: rows are examples, columns are properties | `df.head()` |
| **boolean mask** | A True/False filter that selects rows | `df[df["label"] == 1]` |
| **class** | A blueprint for an object that stores data and behavior together | `class DNASet(...)` |
| **method** | A function that belongs to an object/class | `model.forward(...)` |

### How to read a function

```python
def gc_content(sequence):      # function name + input
    gc = ...                   # work done inside the function
    return gc                  # value sent back
```

Read that as:

> “Given a `sequence`, calculate something called `gc`, then give `gc` back.”

### How to read a class

```python
class ExampleModel(nn.Module):
    def __init__(self):
        ...

    def forward(self, x):
        ...
```

- `__init__` = **what pieces does this object contain?**
- `forward` = **what happens to the input when it moves through the model?**
- `self` = **this particular object**. You normally do not pass it yourself.

Whenever a cell is marked **🔒 RUN ONLY**, focus on the explanation above it rather than every Python detail.


> **New to Python or machine learning?** Work through
> **`Notebook_Start_Here.ipynb`** first — about an hour, and it teaches
> exactly the Python this notebook uses, plus a glossary you can keep open
> in another tab.

## Before you start

**What this notebook is for:** showing where the data came from. You will mostly just run the cells — the dataset is already built and waiting for you.

**What you will leave with:**

1. Data does not arrive clean. Someone makes decisions, and those
   decisions shape every result that follows.
2. **Check your input before you spend an hour on it.** This notebook
   catches a wrong file two different ways.
3. A model can score brilliantly on a dataset that is quietly broken.

**New words you will meet here:** peak, genome, ChIP-seq, background set, blacklist, deduplication

**If you get lost:** you do not need to follow the code here in detail. Read the **bold text** and the section headings, run everything, and move on.

**Time:** about 30 minutes. Run it end to end, then come back to any section that interested you.

## SECTION 1 — Peaks Files and Reference Genomes, Explained

**A PEAKS FILE** (BED / narrowPeak format) is a plain text table telling us
WHERE in the genome a protein was experimentally found bound to DNA:

    chrom   start     end       name   score  strand  signal  pval  qval  summit
    chr1    1000000   1000300   peak1  500    .       12.4    3.2   2.1   150

We only need the first 3 columns: `chrom`, `start`, `end`.

**A REFERENCE GENOME FASTA file** is the actual DNA sequence for an
organism, organized by chromosome:

    >chr1
    NNNNNNNNNNACGTGCATGCATGC...
    >chr2
    ACGTTTGCATGCATGCATGCATGC...

**🔑 Key Terms**
- **Peak** — one experimentally-confirmed binding location
- **BED format** — a simple tab-separated interval format (chrom, start, end, ...)
- **FASTA format** — the standard text format for DNA/protein sequences
- **N** — a placeholder character meaning "this position couldn't be sequenced confidently" (common in centromeres, telomeres, repetitive regions)

**💡 So what?** Given a peaks file + a reference genome, we can look up the
ACTUAL DNA LETTERS at every peak location. Those become our POSITIVE
examples (label=1). We also need NEGATIVE examples (label=0) — DNA windows
where the protein was NOT found bound.

In [ ]:
# 🔒 RUN ONLY
print("=" * 65)
print("  What We Need to Build a Dataset")
print("=" * 65)
print()
print("  Peaks file (.bed / .bed.gz / narrowPeak) -> WHERE binding happens")
print("  Reference genome (.fa / .fa.gz)          -> WHAT the DNA says there")
print()
print("  Output: seqs.txt (one DNA sequence per line)")
print("          labels.txt (1 = binding site, 0 = background)")

## SECTION 1b — Where to Get These Files

📥 Public sources (reference only):

| Resource | Where | Notes |
|---|---|---|
| Peaks files | ENCODE project (encodeproject.org) | Search by transcription factor + cell line, download the "narrowPeak" file |
| Reference genome | UCSC Genome Browser (hgdownload.soe.ucsc.edu/goldenPath) | Pick a species/assembly (e.g. hg38 for human, mm10 for mouse) |

⚠️ Download these to disk BEFORE running this notebook — HPC compute nodes
typically have no internet access.

## SECTION 1c — Trying a Different Target

Everything below works for **any** transcription factor in **any** species,
as long as you have a peaks file and the matching reference genome. Only
Section 3 changes.

### Picking a target, and predicting how hard it will be

This is a good prediction exercise before you run anything. A protein that
binds a long, specific DNA motif is easy to classify from sequence alone. A
protein that binds wherever the chromatin happens to be open leaves much
less sequence signal.

| Target | Type | Expected difficulty | Why |
|---|---|---|---|
| **CTCF** | sequence-specific TF | **easy** (our default) | long, highly conserved ~19 bp motif |
| **REST / NRSF** | sequence-specific TF | easy | very long, very specific motif |
| **GATA1** | sequence-specific TF | moderate | short WGATAR motif, cell-type dependent |
| **JUND** | sequence-specific TF | moderate | short motif, many partners |
| **POLR2A** | RNA polymerase II | hard | binds active promoters, not one motif |
| **H3K4me3** | histone mark | hard | broad chromatin domains, not a motif |
| **EP300** | co-activator | hard | recruited by others, no motif of its own |

If your AUROC drops when you switch from CTCF to POLR2A, that is not a bug.
It is the biology showing up in your metrics — and a much more interesting
result to explain than another 0.95.

### Where the files come from

**Peaks** — ENCODE (`encodeproject.org`):
1. Search the experiment matrix for your target and cell type
2. Filter: *Assay* = TF ChIP-seq, *Genome assembly* = GRCh38
3. In the experiment's file list, take **"IDR thresholded peaks"** or
   **"conservative IDR thresholded peaks"** in `bed narrowPeak` format
4. Note the accession (`ENCFFxxxxxx.bed.gz`) — that is your `PEAKS_FILE`

**Reference genomes** — UCSC (`hgdownload.soe.ucsc.edu/goldenPath/`):

| Species | Assembly | Path |
|---|---|---|
| Human | hg38 | `goldenPath/hg38/bigZips/hg38.fa.gz` |
| Mouse | mm39 | `goldenPath/mm39/bigZips/mm39.fa.gz` |
| Fruit fly | dm6 | `goldenPath/dm6/bigZips/dm6.fa.gz` |
| Zebrafish | danRer11 | `goldenPath/danRer11/bigZips/danRer11.fa.gz` |
| Worm | ce11 | `goldenPath/ce11/bigZips/ce11.fa.gz` |
| Yeast | sacCer3 | `goldenPath/sacCer3/bigZips/sacCer3.fa.gz` |

⚠️ **Two things that will bite you:**

- **The assembly must match.** hg38 peak coordinates on an hg19 genome give
  you real-looking DNA from the wrong places, and a model that learns
  nothing. Check the assembly on the ENCODE page against your FASTA.
- **Chromosome naming must match.** UCSC uses `chr1`; Ensembl uses `1`. If
  `CANONICAL_CHROMS` filtering drops every peak, this is why. Section 5
  prints the count after filtering — if it says 0, check your naming.

⚠️ Download both files **before** running this notebook. Perlmutter compute
nodes have no outbound internet.

In [ ]:
# 🔲 TRY IT — what is already on the shared filesystem?
# (self-contained: this runs before the Section 2 imports)
from pathlib import Path

SHARED_DATA = Path("/global/cfs/cdirs/m4388/projects/project7/data")

for subfolder in ["peaks", "genome"]:
    folder = SHARED_DATA / subfolder
    print(f"{folder}")
    if not folder.exists():
        print("   (not found from this node)")
        continue
    entries = sorted(folder.iterdir())
    if not entries:
        print("   (empty)")
    for entry in entries[:20]:
        size_gb = entry.stat().st_size / 1e9 if entry.is_file() else 0
        print(f"   {entry.name:<40} {size_gb:>8.2f} GB")
    if len(entries) > 20:
        print(f"   ... and {len(entries) - 20} more")
    print()

print("Copy a path from above into PEAKS_FILE / GENOME_FILE in Section 3,")
print("change DATASET_NAME to match, and re-run this notebook.")

## SECTION 2 — Setup
This pipeline needs two genomics-specific libraries:
- **pyfaidx** — fast random access into large FASTA files, without loading
  the whole genome into memory
- **pybedtools** — a Python wrapper around the `bedtools` command-line tool,
  for interval math (subtracting, overlapping regions)

Nothing to edit here — just run it and confirm both libraries load and the
`bedtools` binary is found.

In [ ]:
# What this cell does: Sets the PATH so the bedtools binary can be found, imports the genomics libraries, and 
# runs one tiny test operation to confirm bedtools itself is reachable — 
# this fails loudly with a clear message if bedtools isn't installed/loaded.
# 🔒 RUN ONLY
import os
os.environ["PATH"] = ("/global/cfs/cdirs/m4388/envs/dna-llm/bin:" + os.environ.get("PATH", ""))

import pybedtools
pybedtools.helpers.set_bedtools_path("/global/cfs/cdirs/m4388/envs/dna-llm/bin")

import gzip
import shutil
import random
import json
import pathlib
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    from pyfaidx import Fasta
except ImportError as e:
    raise ImportError("Missing genomics libraries. Install with:\n"
        "  pip install pyfaidx pybedtools pandas tqdm\n"
        "and make sure the 'bedtools' binary is available "
        "(module load bedtools, or conda install -c bioconda bedtools)") from e

# Confirm the bedtools binary itself is reachable
try:
    pybedtools.BedTool.from_dataframe(pd.DataFrame([["chr1", 0, 10]])).sort()
    print("✅ bedtools binary found and working.")
except Exception as e:
    print("⚠️  bedtools binary not found. Run 'module load bedtools' on NERSC,")
    print("   or 'conda install -c bioconda bedtools' in your environment.")
    print(f"   Details: {e}")

## SECTION 3 — Your Task: Configure Your Dataset

This is the ONLY section you need to edit to build a dataset for a
DIFFERENT transcription factor, cell type, or species.

| Variable | Meaning |
|---|---|
| `DATASET_NAME` | Short label used to name the output folder |
| `PEAKS_FILE` | Full path to your local peaks `.bed.gz` file |
| `GENOME_FILE` | Full path to your local reference genome `.fa`/`.fa.gz` file |
| `WINDOW_SIZE` | Length (bp) of each DNA window |
| `RANDOM_NEG_MULTIPLIER` | How many background sequences to draw per positive |
| `SEED` | Random seed — keep fixed for reproducibility |

In [ ]:
# What this cell does: Sets all the configuration variables for this dataset-building run and creates the output directory 
# everything downstream reads from these variables.

# ✏️ EDIT ME
DATASET_NAME = "ctcf_k562"       # rename for your TF/dataset

# ⚠️ PEAKS_FILE must be a ChIP-seq PEAKS file for the protein you want to
#    classify — NOT an exclusion/blacklist file.
#
#    ENCFF356LFX, used here previously, is the ENCODE4 GRCh38 *blacklist*:
#    ~900 problematic regions (centromeres, satellite repeats) that should be
#    EXCLUDED from analysis, not used as positive examples. A real CTCF
#    experiment yields tens of thousands of peaks.
#
#    Get the right file from encodeproject.org:
#      Assay = TF ChIP-seq, Target = CTCF, Biosample = K562, Assembly = GRCh38
#      -> download the "IDR thresholded peaks" file, format bed narrowPeak
#
#    The check cell after Section 5 will tell you if this looks wrong.
PEAKS_FILE = os.path.expanduser(
    "/global/cfs/cdirs/m4388/projects/project7/data/peaks/ENCFF519CXF.bed.gz"
)

GENOME_FILE = os.path.expanduser(
    "/global/cfs/cdirs/m4388/projects/project7/data/genome/hg38.fa.gz"
)

# Optional: an exclusion list to remove from BOTH classes (see Section 5b).
# Set to None to skip. ENCFF356LFX is the correct file for THIS job.
BLACKLIST_FILE = os.path.expanduser(
    "/global/cfs/cdirs/m4388/projects/project7/data/peaks/ENCFF356LFX.bed.gz"
)

WINDOW_SIZE           = 200   # try 200 (default), 100, or 500
RANDOM_NEG_MULTIPLIER = 1     # try 1 (balanced) or 2 (more background than binding)
SEED                  = 42    # keep fixed for reproducibility!

# 🔒 RUN ONLY — canonical chromosomes to keep (avoids alt contigs / scaffolds)
CANONICAL_CHROMS = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}

# All bootcamp notebooks resolve the SAME folder, so a dataset built in
# Notebook 0 is found by Notebooks 1-3 even if you launch them from
# elsewhere. Override by setting the DNA_BOOTCAMP_HOME environment variable.
PROJECT_DIR = Path(os.environ.get("DNA_BOOTCAMP_HOME", ".")).expanduser().resolve()
print("📁 Project directory:", PROJECT_DIR)
OUTPUT_DIR  = PROJECT_DIR / f"{DATASET_NAME}_example"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("🔧 Your dataset configuration:")
print(f"   DATASET_NAME           = {DATASET_NAME}")
print(f"   PEAKS_FILE             = {PEAKS_FILE}")
print(f"   GENOME_FILE             = {GENOME_FILE}")
print(f"   WINDOW_SIZE             = {WINDOW_SIZE}")
print(f"   RANDOM_NEG_MULTIPLIER   = {RANDOM_NEG_MULTIPLIER}")
print(f"   Output directory        = {OUTPUT_DIR}")

## SECTION 4 — Fast Random Access Into a 3-Billion-Letter File

The human genome is ~3.2 billion letters long — far too big to search
naively every time we need a snippet. `pyfaidx.Fasta` builds a small INDEX
file (`.fai`) the first time you open a FASTA, then uses that index to jump
DIRECTLY to any position, without reading the whole file into memory.

**🔑 Key Term**
- **Index (`.fai` file)** — a lookup table mapping "chromosome + position" to a byte offset in the file, enabling instant random access

### 🧩 Function map — reference genome

The next cell defines one helper:

| Function | Input | What it does | Returns |
|---|---|---|---|
| `load_reference_fasta(path)` | path to a FASTA file | Makes the genome searchable without reading the entire genome into RAM at once. If needed, it prepares an uncompressed copy first. | A `pyfaidx.Fasta` object that can fetch DNA by chromosome and coordinates. |

**New word — random access:** jumping directly to one region of a huge file instead of reading from the beginning every time.


In [ ]:
# Decompresses the genome FASTA if needed (pyfaidx requires an uncompressed file) and loads it via pyfaidx, 
# then prints the first few chromosome names found so we can confirm it loaded correctly.
# 🔒 RUN ONLY
def load_reference_fasta(path):
    """Load any reference genome FASTA (gzipped or not) using pyfaidx."""
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Reference genome not found: {path}\n"
            f"Download it before running this notebook (see Section 1b).")

    if path.suffix == ".gz":
        print("🗜️  Decompressing genome FASTA (pyfaidx needs an uncompressed file)...")
        out_path = path.with_suffix("")
        if not out_path.exists():
            with gzip.open(path, "rb") as f_in, open(out_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        path = out_path

    print(f"📄 Loading reference genome: {path}")
    fasta = Fasta(str(path), rebuild=False)
    print(f"✅ Loaded {len(fasta.keys())} sequences (chromosomes/contigs).")
    return fasta

fasta = load_reference_fasta(GENOME_FILE)

print()
print("First 5 sequence names found in this genome file:")
for name in list(fasta.keys())[:5]:
    print(f"   {name}  (length={len(fasta[name]):,} bp)")

## SECTION 5 — Loading and Cleaning the Peaks File

We do THREE things here:
1. **Filter to canonical chromosomes** — genome assemblies also include
   small "alt contigs" and unplaced scaffolds that duplicate or fragment
   real sequence; excluding them keeps sampling simple.
2. **Keep the ORIGINAL peak intervals** — used later so background
   sequences don't accidentally overlap a real binding site.
3. **Build FIXED-WIDTH windows centered on each peak** — real ChIP-seq
   peaks vary in width, but our classifier needs every input to be the
   SAME length, so we center a fixed window on each peak's midpoint.

**🔑 Key Term**
- **Peak midpoint / summit** — the center of a binding region, used as the anchor point for building a fixed-size window

### 🧩 Function map — peaks table

| Function | Input | What it does | Returns |
|---|---|---|---|
| `load_peak_bed(path)` | ChIP-seq peaks BED file | Reads genomic intervals into a table, checks columns, and keeps usable chromosomes/coordinates. | A cleaned pandas `DataFrame`. |

**BED file:** a tab-separated file describing genomic intervals. The first three columns are usually chromosome, start, and end.


In [ ]:
# Reads the peaks BED file, filters to canonical chromosomes only, 
# and builds a fixed-width window centered on each peak's midpoint — printing how many peaks survive each filtering step.

# 🔒 RUN ONLY
def load_peak_bed(peaks_path, window_size, canonical_chroms):
    print(f"📄 Loading peaks file: {peaks_path}")

    df = pd.read_csv(peaks_path, sep="\t", header=None, compression="gzip",
        usecols=[0, 1, 2], names=["chrom", "start", "end"])
    print(f"   Raw peak count            : {len(df):,}")

    df = df[df["chrom"].isin(canonical_chroms)].copy()
    print(f"   After canonical filtering : {len(df):,}")

    # Keep original intervals for background masking (Section 7)
    peak_bt_original = pybedtools.BedTool.from_dataframe(df[["chrom", "start", "end"]])

    # Build fixed-width windows centered on each peak
    half = window_size // 2
    df["center"] = ((df["start"] + df["end"]) // 2).astype(int)
    df["new_start"] = (df["center"] - half).clip(lower=0)
    df["new_end"] = df["new_start"] + window_size

    peak_bt_windows = pybedtools.BedTool.from_dataframe(df[["chrom", "new_start", "new_end"]])

    print(f"✅ Built {len(peak_bt_windows)} fixed-width ({window_size} bp) positive windows.")
    return peak_bt_original, peak_bt_windows

peak_bt_original, peak_bt_windows = load_peak_bed(PEAKS_FILE, WINDOW_SIZE, CANONICAL_CHROMS)

print()
print("First 3 positive windows:")
for iv in list(peak_bt_windows)[:3]:
    print(f"   {iv.chrom}:{iv.start}-{iv.end}")

## SECTION 5b — Is This Actually a Peaks File?

A dataset builder will happily accept any BED file. It cannot tell you that
you handed it the wrong one — the coordinates are valid, the sequences come
out looking like DNA, and the model trains to a high score on a problem you
did not intend to pose.

So we check. Two signals are enough to catch the common mistake:

- **Peak count.** A TF ChIP-seq experiment gives tens of thousands of peaks.
  A few hundred means you are probably holding an exclusion list, a summary
  file, or a single-chromosome subset.
- **Peak width.** Called peaks are typically a few hundred bp. Blacklist
  regions are often many kb — they cover whole repeat arrays.

This is the same habit as the dataset audit in Section 8, applied one step
earlier: check the *input* before you spend an hour building from it.

In [ ]:
# 🔒 RUN ONLY — does PEAKS_FILE look like real ChIP-seq peaks?

peak_widths = [iv.end - iv.start for iv in peak_bt_original]
n_peaks = len(peak_widths)
median_width = int(np.median(peak_widths)) if peak_widths else 0

print("PEAKS FILE SANITY CHECK")
print("-" * 52)
print(f"  Peaks on canonical chromosomes : {n_peaks:,}")
print(f"  Median peak width              : {median_width:,} bp")
print(f"  Widest peak                    : {max(peak_widths):,} bp")
print()

looks_wrong = False

if n_peaks < 5000:
    looks_wrong = True
    print(f"  ⚠️  Only {n_peaks:,} peaks. A TF ChIP-seq experiment usually gives")
    print(f"      20,000-100,000. This may be an exclusion list or a subset.")

if median_width > 2000:
    looks_wrong = True
    print(f"  ⚠️  Median width {median_width:,} bp is very wide for called peaks")
    print(f"      (usually 200-1000 bp). Blacklist regions look like this.")

if looks_wrong:
    print()
    print("  🚨 Check PEAKS_FILE in Section 3 before continuing. If you build")
    print("     a dataset from the wrong file, every downstream metric will")
    print("     describe a task you did not mean to ask.")
else:
    print("  ✅ Peak count and width are in the expected range for ChIP-seq.")

## SECTION 6 — Looking Up the Actual DNA Letters (and a Data-Quality Fix)

For every fixed-width window, we look up the actual DNA letters from the
reference genome. These become our label=1 examples.

**⚠️ Important fix applied in this version:**
Some genome regions couldn't be sequenced confidently and are marked with
the letter **N** instead of A/C/G/T. If we silently kept peak windows
containing N's, and separately (Section 7) DROPPED any background window
containing N's, the two classes would be cleaned inconsistently — a model
could then learn to associate "contains an N" with "is a binding site,"
which is a data ARTIFACT, not real biology.

**The fix:** we now DROP any positive window that contains an N, exactly
matching the rule already used for background windows in Section 7. This
does cost us a small number of real peaks — but it keeps both classes
honest and comparable, which matters far more than dataset size here.

**🔑 Key Term**
- **Data leakage / shortcut artifact** — when a model learns to exploit an
  accidental pattern in how the dataset was built, instead of the real
  signal you actually want it to learn

### 🧩 Function map — turn coordinates into DNA

| Function | Input | What it does | Returns |
|---|---|---|---|
| `extract_sequences(...)` | genome + peak coordinates + desired window size | Looks up the actual A/C/G/T letters around each positive peak. | DNA sequences and their genomic coordinates. |

**Window:** a fixed-length piece of the genome. Using the same window size for every example keeps model inputs comparable.


In [ ]:
# Extracts the DNA sequence for each positive window from the reference genome, 
# DROPPING any window that contains an N — this matches the filtering already applied to background windows, and 
# the cell reports exactly how many peaks were dropped for transparency.

# 🔒 RUN ONLY
def extract_sequences(bed, fasta):
    """Extract DNA sequences for each window, dropping any that contain 'N'.

    Dropping N-containing windows here (instead of keeping them) keeps this
    class treated IDENTICALLY to the background class in Section 7, which
    also excludes N-containing windows. See the markdown cell above for why
    this matters.
    """
    seqs = []
    n_dropped = 0
    for iv in tqdm(bed, desc="Extracting positive sequences"):
        seq = fasta[iv.chrom][iv.start:iv.end].seq.upper()
        if "N" in seq:
            n_dropped += 1
            continue
        seqs.append(seq)

    print(f"\n✅ Extracted {len(seqs):,} positive sequences.")
    print(f"⚠️  Dropped {n_dropped:,} peak window(s) containing 'N' "
          f"(kept for symmetry with the background filtering in Section 7).")
    return seqs

positive_seqs = extract_sequences(peak_bt_windows, fasta)

print(f"   Example: {positive_seqs[0][:60]}...")

## SECTION 7 — Building a Fair Contrast Set

We need DNA windows where the protein was NOT bound, so the model has
something to contrast against. Steps:

1. Build a BED file covering the canonical chromosomes.
2. **Subtract the peak intervals** — leaving only regions with no known
   binding.
3. Randomly sample fixed-width windows from what remains.
4. Skip any window containing `N` (the same rule used for positives in
   Section 6).

**💡 So what?** If we sampled background without subtracting peaks first, we
could label a true binding site as "background" and hand the model
contradictory examples.

### ⚠️ Two bugs that used to live in this cell

Both are worth understanding, because neither one crashes. They just quietly
produce a dataset that answers the wrong question.

**1. `subtract(peaks, A=True)`**

The `-A` flag means *remove the entire feature if there is any overlap at
all*. That is useful when your features are small. Here each feature is an
**entire chromosome** — so any chromosome containing even one peak was
deleted whole. Since peaks land on every chromosome, that removed all of
chr1-chr22, chrX and chrY, and background was then drawn from whatever
scraps remained.

Plain `subtract` (no `-A`) does the intended thing: it carves the peak
intervals out and keeps the rest of each chromosome.

**2. Negatives came from a different chromosome set than positives**

`chrom_lengths` was built from every sequence in the FASTA — including
unplaced scaffolds and alt contigs — while positives were filtered to
canonical chromosomes only. That is a difference between the two classes
that has nothing to do with binding, and a model will happily learn it.

Both are now fixed below, and the cell reports how much of the genome
survived subtraction so you can see it worked.

**🔑 Key Term**
- **Interval subtraction** — computing "everything in set A that does NOT
  overlap set B" (here: genome minus known peaks)

### 🧩 Function map — background examples

The model needs both positive **and** negative examples.

| Function | What it does |
|---|---|
| `generate_background(...)` | Finds genomic regions that do not overlap the known peaks and samples windows from them. |
| `draw_window(...)` | Chooses one valid fixed-width window from an allowed background region. |

Think of the goal as:

```text
positive = region associated with the ChIP-seq target
background = comparable genomic window outside those positive regions
```


In [ ]:
# Builds a genome-wide BED file, subtracts the known peak regions from it, then randomly samples fixed-width windows from the remaining "safe" regions, 
# rejecting any window containing an N, until enough background sequences are collected.
# 🔒 RUN ONLY
def generate_background(peak_bt_original, fasta, n_needed, window_size,
                        seed, output_dir, canonical_chroms):
    random.seed(seed)

    # FIX 2: use the SAME chromosome set the positives were filtered to.
    # Drawing negatives from unplaced scaffolds and alt contigs, while
    # positives come only from canonical chromosomes, is a difference
    # between the classes that has nothing to do with binding.
    chrom_lengths = {chrom: len(fasta[chrom]) for chrom in fasta.keys()
                     if chrom in canonical_chroms}

    if not chrom_lengths:
        raise RuntimeError(
            "No canonical chromosomes found in the FASTA. Check chromosome "
            "naming — UCSC uses 'chr1', Ensembl uses '1'."
        )

    genome_bp = sum(chrom_lengths.values())

    genome_bed_path = output_dir / "genome_intervals.bed"
    with genome_bed_path.open("w") as gf:
        for chrom, length in chrom_lengths.items():
            gf.write(f"{chrom}\t0\t{length}\n")

    genome_bt = pybedtools.BedTool(str(genome_bed_path))

    # FIX 1: NO A=True. Each feature here is a whole chromosome, and -A would
    # delete the entire chromosome on any overlap — leaving nothing to sample
    # from. Plain subtract carves out just the peak intervals.
    nonpeak_bt = genome_bt.subtract(peak_bt_original)
    nonpeak_intervals = list(nonpeak_bt)

    nonpeak_bp = sum(iv.end - iv.start for iv in nonpeak_intervals)
    print(f"   Canonical genome      : {genome_bp:,} bp "
          f"across {len(chrom_lengths)} chromosomes")
    print(f"   After peak subtraction: {nonpeak_bp:,} bp "
          f"({100 * nonpeak_bp / genome_bp:.1f}% remains) "
          f"in {len(nonpeak_intervals):,} intervals")

    if nonpeak_bp < 0.5 * genome_bp:
        print("   ⚠️  More than half the genome was removed. Peaks should")
        print("       cover a small fraction — check PEAKS_FILE.")

    if not nonpeak_intervals:
        raise RuntimeError("No non-peak regions found after subtracting peaks from the genome.")

    def draw_window(iv):
        max_start = iv.end - window_size
        if max_start <= iv.start:
            return None
        s = random.randint(iv.start, max_start)
        return pybedtools.Interval(iv.chrom, s, s + window_size)

    background = []
    attempts, max_attempts = 0, n_needed * 500
    pbar = tqdm(total=n_needed, desc="Sampling background")

    while len(background) < n_needed and attempts < max_attempts:
        attempts += 1
        iv = random.choice(nonpeak_intervals)
        win = draw_window(iv)
        if win is None:
            continue
        seq = fasta[win.chrom][win.start:win.end].seq.upper()
        if "N" in seq:
            continue
        background.append(seq)
        pbar.update(1)
    pbar.close()

    print(f"✅ Generated {len(background):,} background windows "
          f"({attempts:,} attempts, {attempts - len(background):,} rejected).")
    return background

n_background_needed = len(positive_seqs) * RANDOM_NEG_MULTIPLIER
background_seqs = generate_background(
    peak_bt_original, fasta, n_background_needed, WINDOW_SIZE, SEED,
    OUTPUT_DIR, CANONICAL_CHROMS
)

## SECTION 8 — Audit the Dataset *Before* You Write It

Notebooks 1-3 will trust whatever `seqs.txt` contains. So we check it while
it is still just two Python lists, and fix problems here — not after.

| # | Check | Catches |
|---|---|---|
| 1 | Class balance | Skewed classes needing re-weighting later |
| 2 | Sequence length consistency | Bugs in how windows were built |
| 3 | Character validity & N-content by label | Confirms the Section 6 fix worked |
| 4 | Duplicate sequences | The same DNA landing in both train and validation |
| 5 | GC content by class | Real biological signal (CTCF sites are GC-rich) |

Checks 1-4 run below. Check 5 is where Notebook 1 picks up.

### 🧩 Before the audit cells — three data-quality words

- **duplicate:** the exact same DNA sequence appears more than once.
- **conflicting label:** the same DNA sequence appears as both `0` and `1`; that is especially problematic because the model is being told two different answers for identical input.
- **N base:** a genome character meaning the exact nucleotide is unknown at that position.

The audit cells measure these problems **before** anything is written to disk.


In [ ]:
# 🔒 RUN ONLY — assemble the dataset in memory and audit it
import collections

combined_seqs = positive_seqs + background_seqs
combined_labels = [1] * len(positive_seqs) + [0] * len(background_seqs)

labels_arr = np.array(combined_labels)
seq_lengths = np.array([len(s) for s in combined_seqs])

print("CHECK 1: Class Balance")
n_pos = int(labels_arr.sum())
n_neg = len(labels_arr) - n_pos
print(f"  Binding (1)   : {n_pos:,}")
print(f"  Background (0): {n_neg:,}")
print(f"  Ratio         : {n_pos / max(n_neg, 1):.2f} : 1")
print()

print("CHECK 2: Sequence Length Consistency")
expected_len = int(np.median(seq_lengths))
n_wrong_length = int((seq_lengths != expected_len).sum())
print(f"  Expected length (WINDOW_SIZE)     : {expected_len} bp")
print(f"  Sequences with a DIFFERENT length : {n_wrong_length}")
if n_wrong_length > 0:
    print("  ⚠️  Investigate before continuing!")
else:
    print("  ✅ All sequences are the same length.")

In [ ]:
# 🔒 RUN ONLY — CHECK 3: character validity and N-content, by label
all_chars_seen = set("".join(combined_seqs))
unexpected_chars = all_chars_seen - set("ACGT")

print("CHECK 3: Character Validity & N-Content")
print(f"  All characters seen  : {sorted(all_chars_seen)}")
print(f"  Unexpected characters: {unexpected_chars if unexpected_chars else 'None'}")
print()

n_count_by_label = {0: 0, 1: 0}
total_by_label = {0: 0, 1: 0}
for seq, label in zip(combined_seqs, combined_labels):
    total_by_label[label] += 1
    if "N" in seq:
        n_count_by_label[label] += 1

print("  Sequences containing at least one 'N':")
for label in [0, 1]:
    pct = 100 * n_count_by_label[label] / max(total_by_label[label], 1)
    name = "Background" if label == 0 else "Binding"
    print(f"    {name:<10} (label={label}): "
          f"{n_count_by_label[label]} / {total_by_label[label]} ({pct:.1f}%)")

if n_count_by_label[0] == 0 and n_count_by_label[1] == 0:
    print()
    print("  ✅ Both classes are N-free — the Section 6 fix worked as intended.")
else:
    print()
    print("  ⚠️  N's still detected — check that Section 6 ran the patched version.")

In [ ]:
# 🔒 RUN ONLY — CHECK 4: duplicates, and what KIND of duplicate
seq_counts = collections.Counter(combined_seqs)
duplicate_seqs = [seq for seq, c in seq_counts.items() if c > 1]

print("CHECK 4: Duplicate Sequences")
print(f"  Total sequences  : {len(combined_seqs):,}")
print(f"  Unique sequences : {len(seq_counts):,}")
print(f"  Appearing MORE than once: {len(duplicate_seqs)}")
print()

for dup_seq in duplicate_seqs:
    indices = [i for i, s in enumerate(combined_seqs) if s == dup_seq]
    dup_labels = [combined_labels[i] for i in indices]

    print(f"  Duplicate: {dup_seq[:50]}...")
    print(f"    At indices : {indices}")
    print(f"    Labels     : {dup_labels}")

    if len(set(dup_labels)) == 1:
        name = "Binding" if dup_labels[0] == 1 else "Background"
        print(f"    ✅ SAME-LABEL duplicate (both '{name}') — a coincidental repeat,")
        print(f"       e.g. two nearby peaks, or background hitting one region twice.")
    else:
        print(f"    🚨 LABEL CONFLICT — identical DNA labelled BOTH binding AND")
        print(f"       background. The model would get contradictory signal.")
    print()

if not duplicate_seqs:
    print("  ✅ No exact duplicates found.")

## 🎯 Concept: Two Kinds of Duplicate Need Two Different Fixes

Check 4 told you *whether* duplicates exist and *what kind* they are. Those
two kinds are not equally harmful, so they get different treatment:

- **Same-label duplicate** → keep the first copy, drop the rest. Harmless
  repetition; we just do not want it inflating one class.
- **Label conflict** → drop *every* copy. The same DNA cannot be both
  binding and background, and guessing which label is "more correct" would
  bake our guess into the dataset.

We do this now, in memory, so that the files we write are already clean.

### 🧩 Function map — deduplication

`deduplicate_dataset(...)` applies the rule explained above to the assembled examples.

Read the name literally:

```text
de-duplicate dataset
= remove repeated copies while preserving a clean training table
```

The function returns the cleaned dataset plus information about what it removed so the cleanup is auditable.


In [ ]:
# 🔒 RUN ONLY
def deduplicate_dataset(seqs, labels):
    seq_to_labels = collections.defaultdict(list)
    for seq, label in zip(seqs, labels):
        seq_to_labels[seq].append(label)

    kept_seqs, kept_labels = [], []
    n_same_label_dropped = 0
    n_conflict_dropped = 0

    for seq, lbls in seq_to_labels.items():
        if len(set(lbls)) == 1:
            # same-label duplicate(s): keep exactly one copy
            kept_seqs.append(seq)
            kept_labels.append(lbls[0])
            n_same_label_dropped += len(lbls) - 1
        else:
            # label conflict: drop entirely, keep neither
            n_conflict_dropped += len(lbls)

    print(f"Deduplication summary:")
    print(f"  Same-label duplicate copies removed : {n_same_label_dropped}")
    print(f"  Label-conflict sequences removed entirely: {n_conflict_dropped}")
    print(f"  Final sequence count: {len(kept_seqs):,} (was {len(seqs):,})")

    return kept_seqs, kept_labels

# combined_seqs / combined_labels were built in the audit above.

deduped_seqs, deduped_labels = deduplicate_dataset(combined_seqs, combined_labels)

positive_seqs   = [s for s, l in zip(deduped_seqs, deduped_labels) if l == 1]
background_seqs = [s for s, l in zip(deduped_seqs, deduped_labels) if l == 0]

print()
print(f"Final positive count   : {len(positive_seqs):,}")
print(f"Final background count : {len(background_seqs):,}")

## SECTION 9 — Write the Cleaned Dataset, Once

Now — and only now — we write to disk. The format matches EXACTLY what
Notebooks 1-3 expect:

- `seqs.txt` — one DNA sequence per line
- `labels.txt` — one label per line (1 = binding, 0 = background), same order

`dataset_config.json` records how this dataset was built, so a result can
always be traced back to the data that produced it.

**📝 Note:** deduplication can leave the classes slightly uneven (e.g. 909
binding vs 910 background) if the duplicate was a label conflict. That is
fine — the final check below prints the exact ratio.

### 🧩 Function map — save the final dataset

`write_dataset(...)` writes two files in matching row order:

```text
seqs.txt    → DNA sequence on each line
labels.txt  → 0 or 1 for the same line
```

That line-by-line alignment is critical: sequence 25 must correspond to label 25.


In [ ]:
# Writes the final positive and background sequences to seqs.txt/labels.txt in matching order, 
# then saves a config JSON recording every setting and count used to build this dataset.
# 🔒 RUN ONLY
def write_dataset(pos, neg, output_dir):
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    with (output_dir / "seqs.txt").open("w") as sf, (output_dir / "labels.txt").open("w") as lf:
        for s in pos:
            sf.write(s + "\n")
            lf.write("1\n")
        for s in neg:
            sf.write(s + "\n")
            lf.write("0\n")

    print(f"📁 Dataset written to {output_dir}")
    print(f"   seqs.txt   : {len(pos) + len(neg):,} lines")
    print(f"   labels.txt : {len(pos) + len(neg):,} lines")

write_dataset(positive_seqs, background_seqs, OUTPUT_DIR)

config_record = {"dataset_name": DATASET_NAME, "peaks_file": str(PEAKS_FILE),
    "genome_file": str(GENOME_FILE), "window_size": WINDOW_SIZE,
    "random_neg_multiplier": RANDOM_NEG_MULTIPLIER, "seed": SEED,
    "n_positive": len(positive_seqs), "n_background": len(background_seqs),
    "n_containing_windows_excluded_from_both_classes": True}
with open(OUTPUT_DIR / "dataset_config.json", "w") as f:
    json.dump(config_record, f, indent=2)

print(f"✅ Config saved to {OUTPUT_DIR / 'dataset_config.json'}")

In [ ]:
# 🔒 RUN ONLY — final verification: re-read the FILES, not the variables
with open(OUTPUT_DIR / "seqs.txt") as f:
    check_seqs = [line.strip() for line in f if line.strip()]
with open(OUTPUT_DIR / "labels.txt") as f:
    check_labels = [int(line.strip()) for line in f if line.strip()]

print("FINAL VERIFICATION (reading the files from disk)")
print(f"  seqs.txt lines   : {len(check_seqs):,}")
print(f"  labels.txt lines : {len(check_labels):,}")

assert len(check_seqs) == len(check_labels), "seqs and labels are out of sync!"

n_dup = len(check_seqs) - len(set(check_seqs))
n_pos = sum(check_labels)
n_neg = len(check_labels) - n_pos
bad_chars = set("".join(check_seqs)) - set("ACGT")

print(f"  Duplicates       : {n_dup}")
print(f"  Binding (1)      : {n_pos:,}")
print(f"  Background (0)   : {n_neg:,}  ->  {n_pos / max(n_neg, 1):.2f} : 1")
print(f"  Unexpected chars : {bad_chars if bad_chars else 'None'}")
print()

if n_dup == 0 and not bad_chars:
    print("✅ Dataset is clean. Point Notebooks 1-3 at:")
    print(f"   {OUTPUT_DIR}")
else:
    print("⚠️  Something is still wrong — do not train on this yet.")